# VWAP Events 03 — VWAP fade scalp: gross edge, and is it VWAP-specific?

Protocol: `discussions/VWAP_Events/004_claude_ve03_preregistration.md` (fixed before this run).

After a cross, the first bar stretched ≥ k·σ·√15 from VWAP → fade toward VWAP (entry 1 minute later) → exit at target / VWAP touch, stop, or 60 min.
**Primary:** k = 1, exit at VWAP touch, stop 20 bp. **G1:** hour-matched real − placebo > 0. **G2:** real gross lower bound > 7 bp.

In [1]:
import os,sys,json,time,importlib,subprocess
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from dotenv import load_dotenv
pd.set_option('display.width',250);pd.set_option('display.max_columns',40)
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'));load_dotenv(ROOT/'.env')
from data_engineering.modeling import vwap_events as ve, vwap_fade as vf
importlib.reload(vf)
SYNTHETIC=os.getenv('VE03_SYNTHETIC','0')=='1'
ASSETS=['BTC','ETH'];fp=vf.FadeParams()
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='ve03-');OUT=Path(tmp.name)/'03';fp=vf.FadeParams(n_boot=300)
else:
    OUT=ROOT/'notebooks'/'VWAP_Events'/'03'
    BIN=Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent/'Binance'
OUT.mkdir(parents=True,exist_ok=True)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
chk=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_vwap_fade.py')],cwd=str(ROOT),env=env,capture_output=True,text=True)
print(chk.stdout[-300:]);assert chk.returncode==0,'tests failed'
print(json.dumps(vf.registration(fp),indent=1,default=str))

.....                                                                    [100%]
5 passed in 11.03s

{
 "k_grid": [
  1.0,
  2.0
 ],
 "k_primary": 1.0,
 "stretch_m": 15,
 "warmup_min": 30,
 "max_hold": 60,
 "targets": [
  5,
  10,
  20,
  40,
  "vwap"
 ],
 "stops": [
  5,
  10,
  20,
  40
 ],
 "primary_target": "vwap",
 "primary_stop": 20,
 "costs_bp": [
  4.0,
  7.0,
  12.0
 ],
 "tradeable_bp": 7.0,
 "n_boot": 2000,
 "status": "PREREGISTERED_VE004",
 "version": "VE03-0.1.0",
 "gates": {
  "G1": "hour-matched real - placebo mean gross bp, day-bootstrap q2.5 > 0 (primary cell)",
  "G2": "real mean gross bp q2.5 > 7 (primary cell)"
 }
}


In [2]:
T0=time.time();parts=[]
for i,a in enumerate(ASSETS):
    df=vf.synthetic_path_klines(n_days=90,seed=3+i) if SYNTHETIC else pd.read_parquet(BIN/f'{a}USDT_um_1m.parquet',columns=['open_time','high','low','close','volume','quote_volume'])
    parts.append(vf.run_asset(df,a,fp));print(a,f'{time.time()-T0:.0f}s')
R=pd.concat(parts,ignore_index=True)
print('trades simulated:',len(R))

BTC 47s
ETH 92s
trades simulated: 7887924


## Primary decision

In [3]:
res=vf.primary(R,fp);print(json.dumps(res,indent=1,default=float))

{
 "version": "VE03-0.1.0",
 "status": "PREREGISTERED_VE004",
 "n_real": 38174,
 "real_gross_bp": -0.16136577725410461,
 "real_lo": -0.42906290367472555,
 "real_hi": 0.11151867392739072,
 "placebo_gross_bp": 0.02500804141163826,
 "real_minus_placebo_hour_matched": -0.14213407142115078,
 "diff_lo": -0.3938034136521491,
 "diff_hi": 0.10578630556243193,
 "real_minus_placebo_unmatched": -0.1863738004690873,
 "unmatched_lo": -0.4326225511090209,
 "unmatched_hi": 0.06271512453650414,
 "win_rate_real": 0.4752973227851417,
 "G1_vwap_specific": "FAIL",
 "G2_tradeable": "FAIL",
 "real_net_4bp": -4.161365777254105,
 "real_net_7bp": -7.161365777254105,
 "real_net_12bp": -12.161365777254105,
 "verdict": "NO_EDGE"
}


## Full grid (reported, not gated)

In [4]:
G=vf.grid_table(R,fp);G.to_csv(OUT/'03_grid.csv',index=False)
for k in fp.k_grid:
    for mode in ('fade','follow'):
        g=G[(G.k==k)&(G['mode']==mode)]
        print(f'k={k} {mode}: gross bp, rows=target, cols=stop (real | placebo)')
        display(g.pivot_table(index='target',columns=['kind','stop'],values='gross_bp').round(2))
print('win rate vs S/(S+T) (real, k=1, fade, numeric targets):')
w=G[(G.kind=='real')&(G.k==1)&(G['mode']=='fade')&(G.target!='vwap')].copy()
w['theory']=w['stop']/(w['stop']+w.target.astype(float));display(w[['target','stop','win_rate','theory','mean_win','mean_loss','gross_bp','n']].round(3))
print('VWAP-touch exit (real vs placebo):');display(G[G.target=='vwap'][['kind','k','stop','gross_bp','lo','hi','win_rate','mean_win','mean_loss','n']].round(2))

k=1.0 fade: gross bp, rows=target, cols=stop (real | placebo)


kind   placebo                    real                  
stop        5     10    20    40    5     10    20    40
target                                                  
10        0.19 -0.01 -0.12 -0.23  0.15 -0.09 -0.21 -0.29
20        0.51  0.25 -0.00 -0.24  0.46  0.18 -0.07 -0.34
40        0.70  0.45  0.06 -0.26  0.66  0.44  0.11 -0.31
5        -0.34 -0.53 -0.53 -0.48 -0.32 -0.52 -0.52 -0.45
vwap      0.56  0.28  0.03 -0.28  0.49  0.21 -0.16 -0.51

k=1.0 follow: gross bp, rows=target, cols=stop (real | placebo)


kind   placebo                    real                  
stop        5     10    20    40    5     10    20    40
target                                                  
10       -0.20 -0.46 -0.49 -0.54 -0.21 -0.35 -0.39 -0.50
20        0.10 -0.12 -0.13 -0.12  0.11 -0.01 -0.04 -0.14
40        0.27  0.11  0.17  0.22  0.25  0.19  0.29  0.30
5        -0.66 -0.87 -0.84 -0.81 -0.67 -0.82 -0.78 -0.76

k=2.0 fade: gross bp, rows=target, cols=stop (real | placebo)


kind   placebo                    real                  
stop        5     10    20    40    5     10    20    40
target                                                  
10        0.06 -0.14 -0.21 -0.17  0.09 -0.09 -0.16 -0.13
20        0.52  0.35  0.15 -0.05  0.49  0.34  0.12 -0.04
40        0.73  0.58  0.17 -0.14  0.73  0.61  0.20 -0.09
5        -0.58 -0.81 -0.73 -0.57 -0.54 -0.79 -0.74 -0.50
vwap      0.81  0.73  0.42  0.14  0.81  0.71  0.39  0.32

... (2 outputs left out)

win rate vs S/(S+T) (real, k=1, fade, numeric targets):


,target,stop,win_rate,theory,mean_win,mean_loss,gross_bp,n
72,10,5,0.345,0.333,9.936,-4.994,0.154,38174
73,10,10,0.496,0.500,9.901,-9.913,-0.092,38174
74,10,20,0.651,0.667,9.868,-18.983,-0.209,38174
75,10,40,0.756,0.800,9.851,-31.705,-0.289,38174
76,20,5,0.228,0.200,18.967,-4.990,0.463,38174
77,20,10,0.352,0.333,18.724,-9.874,0.180,38174
78,20,20,0.501,0.500,18.542,-18.739,-0.073,38174
79,20,40,0.618,0.667,18.406,-30.714,-0.342,38174
80,40,5,0.157,0.111,31.038,-4.988,0.658,38174
81,40,10,0.258,0.200,30.037,-9.854,0.441,38174


VWAP-touch exit (real vs placebo):


,kind,k,stop,gross_bp,lo,hi,win_rate,mean_win,mean_loss,n
16,placebo,1.0,5,0.56,0.45,0.67,0.22,20.03,-4.99,114630
17,placebo,1.0,10,0.28,0.14,0.44,0.34,20.19,-9.88,114630
18,placebo,1.0,20,0.03,-0.19,0.25,0.48,20.54,-18.82,114630
19,placebo,1.0,40,-0.28,-0.59,0.02,0.60,20.87,-31.45,114630
52,placebo,2.0,5,0.81,0.60,1.04,0.15,32.71,-4.99,49513
53,placebo,2.0,10,0.73,0.42,1.03,0.25,32.22,-9.86,49513
54,placebo,2.0,20,0.42,-0.00,0.83,0.38,31.70,-18.77,49513
55,placebo,2.0,40,0.14,-0.48,0.71,0.50,31.93,-31.63,49513
88,real,1.0,5,0.49,0.35,0.63,0.22,20.05,-4.99,38174
89,real,1.0,10,0.21,0.01,0.40,0.33,20.24,-9.88,38174


## Splits of the primary cell

In [5]:
sel=(R.k==fp.k_primary)&(R['mode']=='fade')&(R.target==str(fp.primary_target))&(R.stop==fp.primary_stop)
P=R[sel].copy();P['period']=np.where(pd.to_datetime(P.day,utc=True)>=pd.Timestamp('2026-04-01',tz='UTC'),'post_2026-04','pre_2026-04')
rows=[]
for keys,g in P.groupby(['kind','asset','anchor','period'],observed=True):
    m,lo,hi,n=vf.boot(g.pnl.to_numpy(),g.day.to_numpy(),fp.n_boot,5)
    rows.append(dict(zip(['kind','asset','anchor','period'],keys),gross_bp=m,lo=lo,hi=hi,n=n,win_rate=float((g.pnl>0).mean())))
S=pd.DataFrame(rows);display(S.round(2));S.to_csv(OUT/'03_primary_splits.csv',index=False)

,kind,asset,anchor,period,gross_bp,lo,hi,n,win_rate
0,placebo,BTC,h4,pre_2026-04,-0.02,-0.25,0.23,38218,0.53
1,placebo,BTC,session,pre_2026-04,-0.18,-0.51,0.16,19605,0.50
2,placebo,ETH,h4,pre_2026-04,0.04,-0.25,0.33,37372,0.45
3,placebo,ETH,session,pre_2026-04,0.29,-0.09,0.66,19435,0.43
4,real,BTC,h4,pre_2026-04,-0.18,-0.53,0.16,12468,0.52
5,real,BTC,session,pre_2026-04,-0.24,-0.69,0.23,6738,0.51
6,real,ETH,h4,pre_2026-04,-0.22,-0.64,0.21,12319,0.44
7,real,ETH,session,pre_2026-04,0.06,-0.60,0.67,6649,0.43


## Save

In [6]:
if not SYNTHETIC:
    (OUT/'03_decision.json').write_text(json.dumps(res,indent=1,default=float))
    (OUT/'03_manifest.json').write_text(json.dumps({'registration':vf.registration(fp),'runtime_s':time.time()-T0,'trades':int(len(R))},indent=1,default=str))
print(f'total {time.time()-T0:.0f}s')

total 99s
